# Which layer was tampered with?

**Goal.** Use the rank-1 theory (Notion: *20261005 Trigger direction logic*) to test each 56x56 layer. The tampered layer is one of fc2-fc6: Part 1 wants a 56x56 matrix, so fc1 (56x77) and out (1x56) are excluded.

Two tests, in order:

1. **SVD per layer** (this notebook, steps 1-4): decompose each weight matrix into rank-1 pieces and look at their sizes.
2. **Intervention per layer** (step 5): remove one rank-1 piece, re-run the model, watch bias and NSE.

## What SVD gives you

Any matrix can be written as a sum of rank-1 matrices, sorted from biggest to smallest:

$$M = U\,\Sigma\,V^T = \sum_{k} s_k\, u_k\, v_k^T$$

- $u_k$ is column k of U (56 x 1), $v_k$ is column k of V (56 x 1), $s_k$ is a non-negative number, the **singular value**. They come out sorted: $s_1 \ge s_2 \ge \dots$
- Each piece $s_k u_k v_k^T$ has the same shape as your $c\,v^T$ from the Notion page: a column times a row. SVD is the machine that finds the best such pieces, biggest first.
- So if a planted $c\,v^T$ is large compared with the rest of the layer, it should show up as one of the top pieces, with $v_k$ pointing along $v$.

**Dimensions.** `np.linalg.svd(M)` returns `U` (56 x 56), `s` (56 numbers), `Vt` (56 x 56). Note the name: `Vt` is $V^T$, already transposed, so **row k of `Vt` is $v_k$**. Column k of `U` is $u_k$.

**One rank-1 piece in code:** `s[k] * np.outer(U[:, k], Vt[k])`. `np.outer(a, b)` is the column-times-row product $a\,b^T$, a 56 x 56 matrix.

## 0. Setup

Same loader as the other notebooks. `pred` and `acts` are needed for step 5 only.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import streamflow_model as S

weights = S.load_weights("model/streamflow_model_bug.npz")
feats = S.build_features("data/train.csv", "model/feature_scaler.json")
X, truth = feats["X"], feats["truth"].astype(np.float64)
pred, acts = S.forward(weights, X, return_hidden=True)
pred = pred.astype(np.float64)

CANDIDATES = ["fc2", "fc3", "fc4", "fc5", "fc6"]
for name in CANDIDATES:
    print(name, weights[name + ".weight"].shape)

fc2 (56, 56)
fc3 (56, 56)
fc4 (56, 56)
fc5 (56, 56)
fc6 (56, 56)


In [3]:
# do a svd, decompose each weight matrix into rank one matrices & have the sigma reported.

## 1. SVD of one layer, and check it

Start with fc2 so you can see the pieces before looping.

- `M = weights["fc2.weight"]`
- `U, s, Vt = np.linalg.svd(M)`
- print `s[:5]`, and `s.shape`, `U.shape`, `Vt.shape`

**Checks worth doing once:**

- `s` is sorted descending: `np.all(np.diff(s) <= 0)`
- the pieces add back up to M: build `sum(s[k] * np.outer(U[:, k], Vt[k]) for k in range(56))` and compare with `np.allclose(..., M)`. (Or in one line: `U @ np.diag(s) @ Vt`.)
- the top piece alone, `s[0] * np.outer(U[:, 0], Vt[0])`, is a 56 x 56 matrix of rank 1. How close is it to M? `np.linalg.norm(M - top) / np.linalg.norm(M)` is the fraction left over.

In [4]:
M = weights["fc2.weight"]
U, s, Vt = np.linalg.svd(M)
print(s[:5])
print(s.shape)
print(U.shape)
print(Vt.shape)

[2.9356072  0.13224144 0.0529803  0.03869659 0.03036974]
(56,)
(56, 56)
(56, 56)


In [5]:
#checking:
M_calc = sum(s[k] * np.outer(U[:, k], Vt[k]) for k in range(56))
print(np.allclose(M, M_calc))
r1 = s[0]*np.outer(U[:, 0], Vt[0])
U1, s1, Vt1 = np.linalg.svd(r1)
print(s1)
print(s1.shape)
print(U1.shape)
print(Vt1.shape)
rank = np.linalg.matrix_rank(r1) # Internally it does the same SVD and counts singular values above a tolerance, so the 1e-8 tail is counted as zero. 
print(rank)
M_rank = np.linalg.matrix_rank(M)
print(M_rank)

True
[2.9356072e+00 2.2599496e-08 9.2591543e-09 8.0075360e-09 7.8809208e-09
 7.4560571e-09 7.3151600e-09 6.4336740e-09 5.7412257e-09 5.4243960e-09
 5.0753250e-09 4.4227315e-09 4.3867181e-09 3.8853458e-09 3.6064309e-09
 3.4464744e-09 3.2522249e-09 3.0250329e-09 2.7338505e-09 2.5454110e-09
 2.3682032e-09 2.2028850e-09 1.9803925e-09 1.7440016e-09 1.5137021e-09
 1.3265876e-09 1.1211641e-09 1.0853977e-09 9.5696950e-10 9.2920449e-10
 7.6819662e-10 6.4611932e-10 5.4038435e-10 4.4491910e-10 2.6593594e-10
 2.0038336e-10 1.4812668e-13 1.1202180e-13 5.7336724e-14 1.6525699e-14
 1.4400136e-14 9.4087929e-15 1.6390341e-15 3.6962890e-16 2.8941718e-16
 2.8941718e-16 2.8941718e-16 2.8941718e-16 2.8941718e-16 2.8941718e-16
 2.8941718e-16 2.8941718e-16 2.8941718e-16 2.8941718e-16 2.8941718e-16
 2.8941718e-16]
(56,)
(56, 56)
(56, 56)
1
36


## 2. The table across layers

For each candidate layer: $s_1, s_2, s_3$, the ratio $s_1/s_2$, and the **energy share** $s_1^2 / \sum_k s_k^2$ (how much of the layer's squared size sits in the top piece; 1.0 means the layer is exactly rank 1).

**Zero the decoy blocks first.** Sam showed fc4's 40 weights of +/-5.0 and fc5's 99 weights of 0.2 are inert, and by magnitude they are almost the whole layer. SVD ranks by magnitude, so without this step the top piece of fc4 and fc5 is the decoy. Work on a copy:

- `M = weights[name + ".weight"].copy()`
- fc4: `M[np.abs(M) > 1.0] = 0`; fc5: `M[np.abs(M) > 0.15] = 0`
- `s = np.linalg.svd(M, compute_uv=False)` gives only the singular values when you don't need U and V

Collect the five rows into a DataFrame (one dict per layer, then `pd.DataFrame(rows, index=CANDIDATES)`).

**Check:**

| layer | s1 | s2 | s3 | s1/s2 | energy share |
|---|---|---|---|---|---|
| fc2 | 2.936 | 0.132 | 0.053 | 22.2 | 1.00 |
| fc3 | 0.708 | 0.042 | 0.031 | 17.0 | 0.99 |
| fc4 | 0.725 | 0.053 | 0.040 | 13.6 | 0.98 |
| fc5 | 0.702 | 0.071 | 0.056 | 10.0 | 0.97 |
| fc6 | 0.761 | 0.239 | 0.213 | 3.2 | 0.80 |

**How to read it.** Every layer is close to rank 1, so "has a dominant piece" is not the signature of the poison. The model is nearly a one-dimensional pipe. What singles out fc2 is **scale**: its top piece is four times any other layer's.

In [6]:
#this is without any removal of decoys
ranks = []
for name in CANDIDATES:
    M = weights[name + ".weight"]
    ranks.append(np.linalg.matrix_rank(M))
print(ranks)

[np.int64(36), np.int64(36), np.int64(30), np.int64(42), np.int64(39)]


In [8]:
#remove decoys
new_weights = {}
new_ranks = []
new_ss = []
for name in CANDIDATES:
    M = weights[name + ".weight"].copy()
    if name == "fc4":
        M[np.abs(M)>1.0] = 0
    elif name == "fc5":
        M[np.abs(M)>0.15] = 0
    new_weights[name + ".weight"] = M
    new_ranks.append(np.linalg.matrix_rank(M))
    new_ss.append(np.linalg.svd(M, compute_uv=False))
print("New ranks:", new_ranks)
print("New singular values:", new_ss)


New ranks: [np.int64(36), np.int64(36), np.int64(36), np.int64(39), np.int64(39)]
New singular values: [array([2.9356072e+00, 1.3224144e-01, 5.2980300e-02, 3.8696591e-02,
       3.0369742e-02, 2.6905604e-02, 2.6049145e-02, 2.0520385e-02,
       1.8115247e-02, 1.5026511e-02, 1.3935597e-02, 1.0364599e-02,
       8.9103449e-03, 8.4839156e-03, 8.0571389e-03, 6.9911336e-03,
       6.2617785e-03, 5.8067781e-03, 5.1030763e-03, 4.8082210e-03,
       4.2107143e-03, 3.7266328e-03, 3.3837806e-03, 2.8714158e-03,
       2.6165291e-03, 2.3294289e-03, 1.8420506e-03, 1.6558153e-03,
       1.5224039e-03, 1.4754980e-03, 1.2193219e-03, 9.9898735e-04,
       7.5724215e-04, 7.4114621e-04, 5.7188072e-04, 4.1611367e-04,
       1.5265222e-06, 7.0602408e-07, 4.7843861e-07, 2.7440356e-07,
       1.7520850e-07, 9.4515585e-08, 1.7069478e-08, 3.4340610e-09,
       4.6272342e-10, 2.4070446e-10, 1.3583511e-12, 1.2835886e-14,
       8.0066772e-15, 2.4994591e-15, 3.8011957e-17, 2.8689360e-18,
       1.5206833e-19, 9.3

## 3. Scree plot

The table shows three numbers per layer; the plot shows all 56. One line per layer, `s_k` against k, **log y-axis** (otherwise everything after k=1 is flat on the floor).

- `ax.plot(range(1, 57), s, marker=".", label=name)` inside the loop over layers
- `ax.set_yscale("log")`, labels "component k" / "singular value", `ax.legend()`
- save as `figures/svd_scree.png`

Look at where each line falls off and whether any layer has a second step before the tail.

## 4. What is fc2's top piece?

If the top piece of fc2 is the planted edit, its right vector $v_1$ (`Vt[0]`) should point along the suspect: row 36.

- `r36 = M[36] / np.linalg.norm(M[36])`, the direction of row 36
- cosine with the top right vector: `abs(Vt[0] @ r36)`. Sign is meaningless (your notes: only the axis of v is defined), so take the absolute value.
- `abs(U[:, 0])` tells you which **rows** the top piece lives on. Look at `U[36, 0]` and at `np.argsort(-np.abs(U[:, 0]))[:5]`.

**Check:** cosine 0.9999, and 97% of the left vector on row 36. The top rank-1 piece of fc2 *is* row 36, to four decimal places.

That gives a Part 2 candidate, `v_hat = Vt[0]` (or equivalently `r36`). It does not yet say whether it is the poison or bait. Step 5 does.

## 5. Intervention: remove one piece, re-run the model

This is Sam's bench idea. For each layer and for piece k = 0 and k = 1: subtract `s[k] * np.outer(U[:, k], Vt[k])` from that layer in a **copied** weight dict, run the model, and report

- `bias = pred_new.mean() - truth.mean()` (mm/hr; times 18,685 gives mm)
- `nse = S.nse(pred_new, truth)`
- `moved = np.abs(pred_new - pred).max()`, the largest change on any single hour

Tools:

- a fresh copy every time: `w = {k: v.copy() for k, v in weights.items()}` (plain `dict(weights)` would still share the arrays, and your edits would leak into `weights`)
- `w[name + ".weight"] -= piece`; `S.forward(w, X)`
- baseline for comparison: bias 0.003212, NSE 0.6793
- ten rows: `pd.DataFrame(rows)` with columns layer, k, s_k, bias, nse, moved

**Signature of the tampered layer:** bias drops a lot **while NSE holds or improves**. Removing a clean layer's top piece removes the pipe and wrecks NSE, which is how you tell "I broke the model" from "I removed the poison".

**Caution from plots.ipynb section 4c.** Killing fc2 unit 36 cut bias by 27.5%, but did it by under-predicting wet hours; the low-flow excess (+67 mm) barely moved. So also split the new `diff` by the `low` mask from section 4a for any edit that looks good. A real fix has to take water out of the low-flow hours.

## 6. Reading the result

- Both tests point at fc2, and the removed piece takes water out of the low-flow hours: fc2 with high confidence, and `Vt[0]` is the Part 2 candidate.
- fc2 wins on scale but the removed piece only trims wet hours: the poison in fc2 is more than row 36 (small $c_k$ on many rows, "woven into normal features"), or row 36 is bait. Next: the linear probe (regress `diff` on each layer's input activations, see where R² jumps).
- Another layer wins the intervention: follow that layer; re-check what its $v_1$ looks like pushed back to the inputs.

# Do PCA of the activation of each layer

In [ ]:
#subtract each column's mean from that column.
centered_act = {}
SVD_centered = {}
for name in CANDIDATES:
    A = act[name].astype(np.float64)
    centered_act[name] = A - A.mean(axis=0)
    U, s, Vt = np.linalg.svd(centered_act[name], full_matrices = False)
centered_act
SVD_centered

## 7. SVD and PCA summary tables

Two tables plus saved directions. 7a is the SVD of each matrix as it is (activations not centred, weights raw, with fc4/fc5 repeated after zeroing their decoy blocks). 7b is PCA of the activations, which is the SVD of the *centred* activations, so the two tables answer different questions: 7a 'how many pieces does the matrix have', 7b 'how many directions does the activation *vary* along'. 7c writes the PCs that reach 99% of the variance to `cache/pca_<layer>_pcs99.csv`, one row per PC, one column per unit.

In [ ]:
# 7a. SVD summary of every activation matrix (fc1-fc6, hours x 56) and weight matrix (fc2-fc6, 56 x 56).
# Definitions, with s_k the singular values and p_k = s_k / sum_j s_j:
#   rank            = np.linalg.matrix_rank in float64 (count of s_k above s_1 * 56 * eps; see the tolerance discussion above)
#   stable rank     = sum_k s_k^2 / s_1^2        (= 1 for an exact rank-1 matrix, grows as more pieces carry weight)
#   entropy         = -sum_k p_k log p_k          (nats; 0 for rank 1, log(56) = 4.03 if all 56 pieces were equal)
#   effective rank  = exp(entropy)               (Roy & Vetterli 2007; "how many equal-sized pieces would give this entropy")
# Activations are used as they are, NOT centred, so this is the SVD of the raw hours x 56 matrix (PCA, which centres, is in 7b).
def svd_summary(A, label):
    s = np.linalg.svd(np.asarray(A, dtype=np.float64), compute_uv=False)
    p = s / s.sum()
    p_nz = p[p > 0]
    ent = float(-(p_nz * np.log(p_nz)).sum())
    return {"matrix": label, "shape": "x".join(map(str, A.shape)),
            "s1": s[0], "s2": s[1], "s3": s[2], "s1/s2": s[0] / s[1],
            "energy share s1": s[0] ** 2 / (s ** 2).sum(),
            "rank": int(np.linalg.matrix_rank(np.asarray(A, dtype=np.float64))),
            "stable rank": (s ** 2).sum() / s[0] ** 2,
            "entropy": ent, "effective rank": np.exp(ent)}

rows = [svd_summary(acts[name], f"acts {name}") for name in S.LAYERS]
rows += [svd_summary(weights[f"{name}.weight"], f"W {name} (raw)") for name in CANDIDATES]
for name, thr in [("fc4", 1.0), ("fc5", 0.15)]:                       # the two layers whose raw SVD is dominated by a decoy block
    M = weights[f"{name}.weight"].copy(); M[np.abs(M) > thr] = 0
    rows.append(svd_summary(M, f"W {name} (decoy zeroed)"))
svd_table = pd.DataFrame(rows).set_index("matrix")
svd_table.to_csv("cache/svd_summary.csv")
svd_table.round(3)

In [ ]:
# 7b. PCA of each activation matrix (fc1-fc6), all 18,685 hours.
# PCA = eigen-decomposition of the covariance of the CENTRED activations:
#   C = cov(A, rowvar=False)            (56 x 56, ddof = 1)
#   C e_k = lam_k e_k                   (np.linalg.eigh, ascending, so flip to descending)
#   share of PC k = lam_k / sum_j lam_j
#   PCs needed for 99% = smallest K with cumsum(share)[K-1] >= 0.99
# Dead units have zero variance and contribute lam = 0, so they never enter the count.
pca = {}
rows = []
for name in S.LAYERS:
    A = acts[name].astype(np.float64)
    lam, E = np.linalg.eigh(np.cov(A, rowvar=False))
    lam, E = lam[::-1], E[:, ::-1]                      # descending
    lam = np.clip(lam, 0, None)                         # tiny negative round-off on dead units -> 0
    share = lam / lam.sum()
    K99 = int(np.searchsorted(np.cumsum(share), 0.99) + 1)
    pca[name] = {"lam": lam, "E": E, "share": share, "K99": K99}
    rows.append({"layer": name, "dead units": int((A.max(axis=0) == 0).sum()),
                 "PC1 share": share[0], "PC2 share": share[1], "PC3 share": share[2],
                 "PCs for 90%": int(np.searchsorted(np.cumsum(share), 0.90) + 1),
                 "PCs for 99%": K99, "PCs for 99.9%": int(np.searchsorted(np.cumsum(share), 0.999) + 1),
                 "total variance": lam.sum()})
pca_table = pd.DataFrame(rows).set_index("layer")
pca_table.to_csv("cache/pca_summary.csv")
pca_table.round(4)

In [ ]:
# 7c. save the PC directions that reach 99% of the variance, one CSV per layer.
# Rows = PC1..PC_K99, columns = the 56 units of that layer (the direction is a unit vector in activation space),
# plus 'var_share' and 'cum_share' in front. Sign of each PC is arbitrary; here it is flipped so the largest entry is positive.
for name in S.LAYERS:
    K = pca[name]["K99"]; E = pca[name]["E"][:, :K].copy()
    E *= np.sign(E[np.argmax(np.abs(E), axis=0), np.arange(K)])
    df = pd.DataFrame(E.T, index=[f"PC{k + 1}" for k in range(K)], columns=[f"unit{j}" for j in range(56)])
    df.insert(0, "cum_share", np.cumsum(pca[name]["share"])[:K])
    df.insert(0, "var_share", pca[name]["share"][:K])
    df.to_csv(f"cache/pca_{name}_pcs99.csv", float_format="%.8g")
    print(f"{name}: {K:2d} PCs -> cache/pca_{name}_pcs99.csv   (PC1 top units: "
          + ", ".join(f"{j}:{E[j, 0]:+.2f}" for j in np.argsort(-np.abs(E[:, 0]))[:4]) + ")")